In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: echonext-shd-interactive-plot
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path("..")

# Load EchoNext per-task results
echonext_df = pd.read_csv(ROOT / "results/comprehensive_latest_48_models/tables/echonext_shd_per_task.csv")

# Filter key diagnostic tasks
selected_tasks = [
    "lvef_lte_45", 
    "aortic_stenosis_moderate_or_greater", 
    "composite_shd_moderate_or_greater"
]
sub_df = echonext_df[echonext_df['task'].isin(selected_tasks)].copy()
sub_df['family'] = sub_df['model_id'].str.split("__").str[0]

# Interactive Scatter Plot: AUROC vs F1 Score grouped by Family & Task
fig_shd = px.scatter(
    sub_df, 
    x="auroc", 
    y="f1", 
    color="family", 
    symbol="task",
    hover_data=["model_id", "sensitivity", "specificity", "brier"],
    title="EchoNext Structural Heart Disease Diagnostic Performance (AUROC vs F1)",
    labels={"auroc": "Downstream Classifier AUROC", "f1": "Downstream F1-Score at 0.5 Threshold"}
)

fig_shd.update_layout(
    template="plotly_dark",
    height=500,
    margin=dict(l=20, r=20, t=50, b=20)
)
fig_shd.show()

In [3]:
#| label: echonext-probability-drift-boxplots
import plotly.express as px

# Compute probability drift delta from reference
echonext_df['family'] = echonext_df['model_id'].str.split("__").str[0]
echonext_df['prob_brier_drift'] = echonext_df['brier']

fig_drift_box = px.box(
    echonext_df, 
    x="task", 
    y="auroc", 
    color="family",
    title="EchoNext 12-Task Diagnostic AUROC Variation Across Model Families",
    labels={"auroc": "Task AUROC Score", "task": "Structural Heart Disease Task"}
)
fig_drift_box.update_layout(
    template="plotly_dark", 
    height=480, 
    xaxis_tickangle=-45,
    margin=dict(l=20, r=20, t=50, b=80)
)
fig_drift_box.show()

In [4]:
#| label: echonext-classifier-table
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
tasks = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/tables/echonext_shd_per_task.csv"
)

pd.DataFrame({
    "quantity": ["rows", "models", "tasks", "records per row",
                 "minimum positives", "maximum positives"],
    "value": [len(tasks), tasks.model_id.nunique(), tasks.task.nunique(),
              int(tasks.n.min()), int(tasks.support_positive.min()),
              int(tasks.support_positive.max())]
})

,quantity,value
0,rows,576
1,models,48
2,tasks,12
3,records per row,5442
4,minimum positives,20
5,maximum positives,2318


In [5]:
#| label: echonext-macro-summary
#| tbl-cap: Descriptive macro classifier metrics across 12 EchoNext tasks.
macro = (
    tasks.groupby("model_id", as_index=False)
    .agg(macro_auroc=("auroc", "mean"),
         macro_ap=("average_precision", "mean"),
         macro_f1=("f1", "mean"),
         macro_brier=("brier", "mean"),
         macro_ece=("ece", "mean"))
)
macro.sort_values("macro_auroc", ascending=False).head(12)

,model_id,macro_auroc,macro_ap,macro_f1,macro_brier,macro_ece
10,ecgaim__e1c0m1d0__s42,0.780984,0.284399,0.289663,0.174302,0.276164
27,msvae__e1c0m1d1__s42,0.776604,0.277770,0.289285,0.160585,0.241212
0,ecgaim__e0c0m0d0__s42,0.775997,0.282799,0.290308,0.177045,0.284286
11,ecgaim__e1c0m1d1__s42,0.775573,0.285396,0.291095,0.175033,0.281892
3,ecgaim__e0c0m1d1__s42,0.775538,0.279072,0.283194,0.183507,0.289990
1,ecgaim__e0c0m0d1__s42,0.775372,0.283891,0.289576,0.180691,0.292005
25,msvae__e1c0m0d1__s42,0.775075,0.274108,0.273806,0.195070,0.304499
12,ecgaim__e1c1m0d0__s42,0.774931,0.286427,0.287565,0.183956,0.299431
14,ecgaim__e1c1m1d0__s42,0.774653,0.285284,0.291904,0.177199,0.287304
9,ecgaim__e1c0m0d1__s42,0.774462,0.283443,0.291978,0.175448,0.282816


In [6]:
#| label: echonext-reference-and-anchor-fidelity
#| tbl-cap: Original-waveform reference and reconstructed anchor metrics recomputed from locked per-record EchoNext artifacts.
import io
import re
import tarfile
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

ARCHIVE = (
    ROOT / "results/comprehensive_latest_48_models/"
           "referenced_artifacts/factorial_v4.tar.gz"
)
reference_member = "factorial_v4/echonext_reference_shd.parquet"
archive_prefix = "factorial_v4/echonext_per_record/"

anchor_ids = [
    "unet__e1c0m0d0__s42", "unet__e1c1m1d1__s42",
    "msvae__e1c0m0d0__s42", "msvae__e1c1m1d1__s42",
    "ecgaim__e1c0m0d0__s42", "ecgaim__e1c1m1d1__s42",
]

target_members = {reference_member}
for model_id in anchor_ids:
    target_members.add(f"{archive_prefix}{model_id}__echonext_shd.parquet")
    target_members.add(f"{archive_prefix}{model_id}.parquet")
archived = {}
with tarfile.open(ARCHIVE, mode="r:gz") as bundle:
    for member in bundle:
        if member.name in target_members:
            payload = bundle.extractfile(member)
            archived[member.name] = pd.read_parquet(io.BytesIO(payload.read()))
missing_members = sorted(target_members - set(archived))
if missing_members:
    raise FileNotFoundError(f"Missing required archive members: {missing_members}")

reference_records = archived[reference_member].sort_values("row_index")
reference_label = np.stack(reference_records.labels)
reference_probability = np.stack(reference_records.probabilities)
task_names = (
    tasks[["task", "task_index"]].drop_duplicates()
    .sort_values("task_index").task.tolist()
)

def fixed_bin_ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum(
        (index == b).mean() * abs(p[index == b].mean() - y[index == b].mean())
        for b in range(bins) if (index == b).any()
    )

reference_per_task = {}
for task_index, task in enumerate(task_names):
    y = reference_label[:, task_index]
    p = reference_probability[:, task_index]
    reference_per_task[task] = {
        "auroc": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "brier": brier_score_loss(y, p),
        "ece": fixed_bin_ece(y, p),
        "support_positive": int(y.sum()),
    }
reference_macro = {
    metric: float(np.mean([row[metric] for row in reference_per_task.values()]))
    for metric in ("auroc", "average_precision", "brier", "ece")
}

echo_results = {
    "shd_reference": {"clinical": {
        "macro": reference_macro,
        "per_task": reference_per_task,
    }},
    "models": {},
}
for model_id in anchor_ids:
    per_task = tasks.query("model_id == @model_id").set_index("task")
    clean = archived[f"{archive_prefix}{model_id}__echonext_shd.parquet"].query(
        "condition == 'clean'"
    ).sort_values("row_index")
    reconstructed_probability = np.stack(clean.probabilities)
    mask_match = re.search(r"__e([01])c([01])m([01])d([01])__", model_id)
    factorial_mask = "".join(mask_match.groups())
    echo_results["models"][model_id] = {
        "family": model_id.split("__", 1)[0],
        "factorial_mask": factorial_mask,
        "shd_clinical": {
            "macro": {
                "auroc": float(per_task.auroc.mean()),
                "average_precision": float(per_task.average_precision.mean()),
                "brier": float(per_task.brier.mean()),
                "ece": float(per_task.ece.mean()),
            },
            "per_task": per_task.to_dict(orient="index"),
        },
        "shd_probability_fidelity": {"macro": {
            "probability_mae": float(np.abs(reconstructed_probability - reference_probability).mean()),
            "probability_pearson": float(np.corrcoef(reconstructed_probability.ravel(), reference_probability.ravel())[0, 1]),
            "threshold_agreement": float(((reconstructed_probability >= 0.5) == (reference_probability >= 0.5)).mean()),
        }},
    }

anchor_rows = [{
    "input": "original 12-lead",
    "family": "reference",
    "loss": "not applicable",
    "macro_AUROC": reference_macro["auroc"],
    "macro_AP": reference_macro["average_precision"],
    "macro_Brier": reference_macro["brier"],
    "macro_ECE": reference_macro["ece"],
    "probability_MAE_vs_original": 0.0,
    "probability_Pearson_vs_original": 1.0,
    "threshold_agreement_vs_original": 1.0,
}]
for model_id in anchor_ids:
    model = echo_results["models"][model_id]
    clinical = model["shd_clinical"]["macro"]
    fidelity = model["shd_probability_fidelity"]["macro"]
    anchor_rows.append({
        "input": "reconstructed 12-lead",
        "family": model["family"],
        "loss": "MSE-only" if model["factorial_mask"] == "1000" else "full",
        "macro_AUROC": clinical["auroc"],
        "macro_AP": clinical["average_precision"],
        "macro_Brier": clinical["brier"],
        "macro_ECE": clinical["ece"],
        "probability_MAE_vs_original": fidelity["probability_mae"],
        "probability_Pearson_vs_original": fidelity["probability_pearson"],
        "threshold_agreement_vs_original": fidelity["threshold_agreement"],
    })
anchor_fidelity = pd.DataFrame(anchor_rows)
print(f"Recomputed from locked record-level archive: {ARCHIVE.resolve()}")
anchor_fidelity

Recomputed from locked record-level archive: /home/mithunmanivannan/projects/benchmarking_loss_functions_ecg_reconstruction/results/comprehensive_latest_48_models/referenced_artifacts/factorial_v4.tar.gz


,input,family,loss,macro_AUROC,macro_AP,macro_Brier,macro_ECE,probability_MAE_vs_original,probability_Pearson_vs_original,threshold_agreement_vs_original
0,original 12-lead,reference,not applicable,0.802625,0.311546,0.163934,0.251601,0.000000,1.000000,1.000000
1,reconstructed 12-lead,unet,MSE-only,0.744775,0.238297,0.217232,0.349217,0.183932,0.564233,0.713662
2,reconstructed 12-lead,unet,full,0.723323,0.229388,0.237674,0.372992,0.206554,0.506999,0.666605
3,reconstructed 12-lead,msvae,MSE-only,0.771845,0.275559,0.191681,0.304458,0.114482,0.816035,0.840040
4,reconstructed 12-lead,msvae,full,0.762317,0.267238,0.237647,0.381970,0.172136,0.714245,0.741348
5,reconstructed 12-lead,ecgaim,MSE-only,0.772212,0.281604,0.181161,0.288867,0.094928,0.867425,0.887526
6,reconstructed 12-lead,ecgaim,full,0.770991,0.283758,0.177984,0.289402,0.090676,0.881542,0.900328


In [7]:
#| label: echonext-per-task-reference-deltas
#| tbl-cap: Per-task reconstructed-minus-original classifier changes for six anchor models.
reference_tasks = (
    pd.DataFrame(echo_results["shd_reference"]["clinical"]["per_task"])
      .T.rename_axis("task").reset_index()
).rename(columns={
    "auroc": "reference_auroc",
    "average_precision": "reference_ap",
    "brier": "reference_brier",
    "ece": "reference_ece",
})
delta_rows = []
for model_id in anchor_ids:
    model = echo_results["models"][model_id]
    reconstructed = (
        pd.DataFrame(model["shd_clinical"]["per_task"])
          .T.rename_axis("task").reset_index()
    )
    joined = reconstructed.merge(
        reference_tasks[
            ["task", "reference_auroc", "reference_ap",
             "reference_brier", "reference_ece"]
        ],
        on="task",
        validate="one_to_one",
    )
    joined["model_id"] = model_id
    joined["delta_auroc"] = joined["auroc"] - joined["reference_auroc"]
    joined["delta_ap"] = joined["average_precision"] - joined["reference_ap"]
    joined["delta_brier"] = joined["brier"] - joined["reference_brier"]
    joined["delta_ece"] = joined["ece"] - joined["reference_ece"]
    delta_rows.append(joined)
task_deltas = pd.concat(delta_rows, ignore_index=True)
task_deltas[[
    "model_id", "task", "support_positive",
    "reference_auroc", "auroc", "delta_auroc",
    "reference_ap", "average_precision", "delta_ap",
    "delta_brier", "delta_ece",
]].sort_values(["model_id", "delta_auroc"])

,model_id,task,support_positive,reference_auroc,auroc,delta_auroc,reference_ap,average_precision,delta_ap,delta_brier,delta_ece
54,ecgaim__e1c0m0d0__s42,pulmonary_regurgitation_moderate_or_greater,20,0.831649,0.756962,-0.074686,0.116474,0.062689,-0.053785,0.027611,0.067551
51,ecgaim__e1c0m0d0__s42,aortic_regurgitation_moderate_or_greater,66,0.739634,0.673115,-0.066519,0.032479,0.04033,0.007851,-0.005618,0.008746
58,ecgaim__e1c0m0d0__s42,tr_max_gte_32,375,0.753589,0.722238,-0.031351,0.210532,0.162532,-0.048,0.021757,0.049754
59,ecgaim__e1c0m0d0__s42,shd_moderate_or_greater,2318,0.819712,0.790027,-0.029685,0.788800,0.754055,-0.034745,0.013128,0.0189
48,ecgaim__e1c0m0d0__s42,lvef_lte_45,962,0.851662,0.824773,-0.026889,0.598873,0.546985,-0.051888,0.010722,0.021221
...,...,...,...,...,...,...,...,...,...,...,...
12,unet__e1c1m1d1__s42,lvef_lte_45,962,0.851662,0.79865,-0.053012,0.598873,0.480334,-0.118539,0.106618,0.186251
13,unet__e1c1m1d1__s42,lvwt_gte_13,1061,0.734301,0.686658,-0.047643,0.371502,0.318257,-0.053246,0.15009,0.230602
16,unet__e1c1m1d1__s42,mitral_regurgitation_moderate_or_greater,337,0.806109,0.765876,-0.040232,0.222049,0.16546,-0.05659,0.113433,0.185381
15,unet__e1c1m1d1__s42,aortic_regurgitation_moderate_or_greater,66,0.739634,0.720821,-0.018813,0.032479,0.031463,-0.001016,0.237773,0.297168


In [8]:
#| label: echonext-record-level-load
#| tbl-cap: Record-level EchoNext archive integrity and join audit.
import io
import tarfile

ARCHIVE = (
    ROOT / "results/comprehensive_latest_48_models/"
           "referenced_artifacts/factorial_v4.tar.gz"
)
archive_prefix = "factorial_v4/echonext_per_record/"
reference_member = "factorial_v4/echonext_reference_shd.parquet"
target_members = {reference_member}
for model_id in anchor_ids:
    target_members.add(f"{archive_prefix}{model_id}__echonext_shd.parquet")
    target_members.add(f"{archive_prefix}{model_id}.parquet")

archived = {}
with tarfile.open(ARCHIVE, mode="r:gz") as bundle:
    for member in bundle:
        if member.name in target_members:
            payload = bundle.extractfile(member)
            archived[member.name] = pd.read_parquet(io.BytesIO(payload.read()))

missing_members = sorted(target_members - set(archived))
if missing_members:
    raise FileNotFoundError(f"Missing required archive members: {missing_members}")

reference_records = archived[reference_member].sort_values("row_index")
test_metadata = pd.read_csv(
    ROOT / "data/echonext/echonext_metadata_100k.csv"
).query("split == 'test'")

archive_gate = pd.DataFrame({
    "quantity": [
        "archive exists", "required Parquet members found",
        "reference records", "unique reference ECG keys",
        "test metadata records", "reference keys matched to metadata",
        "reference label vectors with 12 tasks",
        "reference probability vectors with 12 tasks",
    ],
    "value": [
        ARCHIVE.is_file(), len(archived), len(reference_records),
        reference_records.ecg_key.nunique(), len(test_metadata),
        reference_records.ecg_key.isin(test_metadata.ecg_key).sum(),
        reference_records.labels.map(len).eq(12).sum(),
        reference_records.probabilities.map(len).eq(12).sum(),
    ],
})
print(f"Loaded record-level classifier outputs from: {ARCHIVE.resolve()}")
archive_gate

Loaded record-level classifier outputs from: /home/mithunmanivannan/projects/benchmarking_loss_functions_ecg_reconstruction/results/comprehensive_latest_48_models/referenced_artifacts/factorial_v4.tar.gz


,quantity,value
0,archive exists,True
1,required Parquet members found,13
2,reference records,5442
3,unique reference ECG keys,5442
4,test metadata records,5442
5,reference keys matched to metadata,5442
6,reference label vectors with 12 tasks,5442
7,reference probability vectors with 12 tasks,5442


In [9]:
#| label: echonext-record-drift-tails
#| tbl-cap: Paired record-level probability drift relative to the original 12-lead classifier.
task_names = list(echo_results["shd_reference"]["clinical"]["per_task"])
reference_probability = np.stack(reference_records.probabilities)
reference_label = np.stack(reference_records.labels)
reference_lookup = reference_records[["row_index", "ecg_key"]].copy()
reference_lookup["reference_probability"] = list(reference_probability)
reference_lookup["labels"] = list(reference_label)

rng = np.random.default_rng(20260731)
record_frames = []
for model_id in anchor_ids:
    shd_member = f"{archive_prefix}{model_id}__echonext_shd.parquet"
    clean = (
        archived[shd_member].query("condition == 'clean'")
        .sort_values("row_index")
    )
    if len(clean) != len(reference_records):
        raise ValueError(f"{model_id}: clean row count does not match reference")
    if not np.array_equal(clean.ecg_key.to_numpy(),
                          reference_records.ecg_key.to_numpy()):
        raise ValueError(f"{model_id}: ECG order/key mismatch")
    reconstructed_probability = np.stack(clean.probabilities)
    absolute_delta = np.abs(reconstructed_probability - reference_probability)
    threshold_delta = (
        (reconstructed_probability >= 0.5) !=
        (reference_probability >= 0.5)
    )
    frame = reference_records[["row_index", "ecg_key"]].copy()
    frame["model_id"] = model_id
    frame["family"] = echo_results["models"][model_id]["family"]
    frame["loss"] = (
        "MSE-only"
        if echo_results["models"][model_id]["factorial_mask"] == "1000"
        else "full"
    )
    frame["probability_drift"] = absolute_delta.mean(axis=1)
    frame["threshold_disagreement"] = threshold_delta.mean(axis=1)
    frame["largest_task_index"] = absolute_delta.argmax(axis=1)
    frame["largest_task"] = frame["largest_task_index"].map(
        lambda index: task_names[index]
    )
    frame["largest_task_drift"] = absolute_delta.max(axis=1)
    frame["reference_probability_vector"] = list(reference_probability)
    frame["reconstructed_probability_vector"] = list(
        reconstructed_probability
    )
    record_frames.append(frame)

record_drift = (
    pd.concat(record_frames, ignore_index=True)
    .merge(
        test_metadata[[
            "ecg_key", "patient_key", "age_at_ecg", "sex",
            "race_ethnicity", "location_setting", "acquisition_year",
        ]],
        on="ecg_key", how="left", validate="many_to_one",
    )
)

tail_rows = []
for model_id, group in record_drift.groupby("model_id", sort=False):
    patient_means = group.groupby("patient_key").probability_drift.mean()
    bootstrap = np.array([
        rng.choice(patient_means.to_numpy(), len(patient_means), replace=True).mean()
        for _ in range(1000)
    ])
    tail_rows.append({
        "model_id": model_id,
        "patients": patient_means.size,
        "record_mean": group.probability_drift.mean(),
        "patient_weighted_mean": patient_means.mean(),
        "patient_bootstrap_95%_low": np.quantile(bootstrap, 0.025),
        "patient_bootstrap_95%_high": np.quantile(bootstrap, 0.975),
        "median": group.probability_drift.median(),
        "p90": group.probability_drift.quantile(0.90),
        "p95": group.probability_drift.quantile(0.95),
        "p99": group.probability_drift.quantile(0.99),
        "maximum": group.probability_drift.max(),
        "threshold_disagreement": group.threshold_disagreement.mean(),
    })
drift_tails = pd.DataFrame(tail_rows)
drift_tails

,model_id,patients,record_mean,patient_weighted_mean,patient_bootstrap_95%_low,patient_bootstrap_95%_high,median,p90,p95,p99,maximum,threshold_disagreement
0,unet__e1c0m0d0__s42,5442,0.183932,0.183932,0.181668,0.186055,0.181787,0.296217,0.325790,0.408020,0.592689,0.286338
1,unet__e1c1m1d1__s42,5442,0.206554,0.206554,0.203984,0.209231,0.199241,0.326965,0.371234,0.481785,0.625134,0.333395
2,msvae__e1c0m0d0__s42,5442,0.114482,0.114482,0.112365,0.116708,0.092567,0.230873,0.273424,0.346470,0.621085,0.159960
3,msvae__e1c1m1d1__s42,5442,0.172136,0.172136,0.169402,0.174726,0.157544,0.316359,0.353684,0.432864,0.633788,0.258652
4,ecgaim__e1c0m0d0__s42,5442,0.094928,0.094928,0.093206,0.096739,0.078970,0.189475,0.230032,0.310035,0.536053,0.112474
5,ecgaim__e1c1m1d1__s42,5442,0.090676,0.090676,0.088856,0.092653,0.071928,0.188113,0.227675,0.294938,0.537689,0.099672


In [10]:
#| label: echonext-composite-reliability
#| fig-cap: Reliability of the frozen EchoNext composite-SHD output for original and reconstructed waveforms. Marker size is proportional to bin support.
import plotly.graph_objects as go

composite_index = task_names.index("shd_moderate_or_greater")
composite_labels = reference_label[:, composite_index]
bin_edges = np.linspace(0, 1, 11)

def reliability_rows(name, probabilities):
    bin_id = np.clip(np.digitize(probabilities, bin_edges) - 1, 0, 9)
    rows = []
    for index in range(10):
        selected = bin_id == index
        if selected.any():
            rows.append({
                "input": name,
                "bin": index,
                "n": selected.sum(),
                "mean_probability": probabilities[selected].mean(),
                "observed_prevalence": composite_labels[selected].mean(),
            })
    return rows

reliability = reliability_rows(
    "original 12-lead", reference_probability[:, composite_index]
)
calibration_rows = []
for name, probabilities in [
    ("original 12-lead", reference_probability),
    *[
        (
            model_id,
            np.stack(
                archived[
                    f"{archive_prefix}{model_id}__echonext_shd.parquet"
                ].query("condition == 'clean'").sort_values("row_index").probabilities
            ),
        )
        for model_id in anchor_ids
    ],
]:
    task_probability = probabilities[:, composite_index]
    rel = pd.DataFrame(reliability_rows(name, task_probability))
    reliability.extend(rel.to_dict("records"))
    calibration_rows.append({
        "input": name,
        "Brier": np.mean((task_probability - composite_labels) ** 2),
        "ECE_10_equal_width": np.average(
            np.abs(rel.mean_probability - rel.observed_prevalence),
            weights=rel.n,
        ),
        "maximum_bin_gap": np.abs(
            rel.mean_probability - rel.observed_prevalence
        ).max(),
    })

reliability = pd.DataFrame(reliability).drop_duplicates(
    ["input", "bin"], keep="first"
)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
    line=dict(color="black", dash="dash"),
))
for name, group in reliability.groupby("input", sort=False):
    fig.add_trace(go.Scatter(
        x=group.mean_probability, y=group.observed_prevalence,
        mode="lines+markers", name=name,
        marker=dict(size=5 + 12 * np.sqrt(group.n / group.n.max())),
        customdata=group[["n", "bin"]],
        hovertemplate=(
            "mean predicted=%{x:.3f}<br>observed=%{y:.3f}"
            "<br>n=%{customdata[0]}<br>bin=%{customdata[1]}<extra></extra>"
        ),
    ))
fig.update_layout(
    xaxis_title="Mean predicted probability",
    yaxis_title="Observed composite-SHD prevalence",
    legend_title="Waveform input",
)
fig.show()
pd.DataFrame(calibration_rows)

,input,Brier,ECE_10_equal_width,maximum_bin_gap
0,original 12-lead,0.169012,0.015307,0.059778
1,unet__e1c0m0d0__s42,0.229484,0.162818,0.248889
2,unet__e1c1m1d1__s42,0.249290,0.210521,0.289625
3,msvae__e1c0m0d0__s42,0.191744,0.069127,0.146480
4,msvae__e1c1m1d1__s42,0.225655,0.176922,0.244740
5,ecgaim__e1c0m0d0__s42,0.182139,0.034206,0.069584
6,ecgaim__e1c1m1d1__s42,0.182402,0.044061,0.196695


In [11]:
#| label: echonext-subgroup-probability-drift
#| tbl-cap: Largest adequately supported subgroup probability-drift estimates within each anchor model.
record_drift["age_band"] = pd.cut(
    record_drift.age_at_ecg,
    bins=[0, 40, 55, 65, 75, 120],
    labels=["≤40", "41–55", "56–65", "66–75", ">75"],
    include_lowest=True,
)
subgroup_rows = []
for model_id, model_group in record_drift.groupby("model_id", sort=False):
    for dimension in ["sex", "age_band", "race_ethnicity", "location_setting"]:
        for level, group in model_group.groupby(dimension, observed=True):
            subgroup_rows.append({
                "model_id": model_id,
                "dimension": dimension,
                "group": str(level),
                "records": len(group),
                "patients": group.patient_key.nunique(),
                "mean_probability_drift": group.probability_drift.mean(),
                "p95_probability_drift": group.probability_drift.quantile(0.95),
                "threshold_disagreement": group.threshold_disagreement.mean(),
            })
subgroup_drift = pd.DataFrame(subgroup_rows)
supported_subgroups = subgroup_drift.query(
    "records >= 100 and patients >= 100"
)
(
    supported_subgroups.sort_values(
        ["model_id", "mean_probability_drift"],
        ascending=[True, False],
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(6)
)

,model_id,dimension,group,records,patients,mean_probability_drift,p95_probability_drift,threshold_disagreement
83,ecgaim__e1c0m0d0__s42,location_setting,outpatient,1059,1059,0.108136,0.245376,0.091596
70,ecgaim__e1c0m0d0__s42,age_band,≤40,629,629,0.105898,0.240312,0.098039
84,ecgaim__e1c0m0d0__s42,location_setting,procedural,209,209,0.098813,0.233497,0.136364
78,ecgaim__e1c0m0d0__s42,race_ethnicity,other,457,457,0.098303,0.220399,0.125638
71,ecgaim__e1c0m0d0__s42,age_band,41–55,1026,1026,0.097611,0.227856,0.090237
77,ecgaim__e1c0m0d0__s42,race_ethnicity,hispanic,1649,1649,0.096845,0.226881,0.099707
87,ecgaim__e1c1m1d1__s42,age_band,≤40,629,629,0.109420,0.250689,0.088368
100,ecgaim__e1c1m1d1__s42,location_setting,outpatient,1059,1059,0.108101,0.247502,0.082704
88,ecgaim__e1c1m1d1__s42,age_band,41–55,1026,1026,0.097301,0.235455,0.081059
94,ecgaim__e1c1m1d1__s42,race_ethnicity,hispanic,1649,1649,0.093987,0.234135,0.086568


In [12]:
#| label: echonext-worst-records
#| tbl-cap: Highest paired classifier-drift records for each prespecified anchor model.
morphology_frames = []
for model_id in anchor_ids:
    member = f"{archive_prefix}{model_id}.parquet"
    morphology = archived[member].query("condition == 'clean'").copy()
    morphology = morphology.rename(columns={"ecg_id": "row_index"})
    morphology["model_id"] = model_id
    morphology_frames.append(morphology)
morphology = pd.concat(morphology_frames, ignore_index=True)
record_drift = record_drift.merge(
    morphology[[
        "model_id", "row_index", "pearson", "rmse", "mae",
        "derivative_mse",
    ]],
    on=["model_id", "row_index"],
    how="left",
    validate="one_to_one",
)

worst_records = (
    record_drift.sort_values(
        ["model_id", "probability_drift"], ascending=[True, False]
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(3)
    .copy()
)
worst_records["reference_probability_on_largest_task"] = worst_records.apply(
    lambda row: row.reference_probability_vector[row.largest_task_index],
    axis=1,
)
worst_records["reconstructed_probability_on_largest_task"] = worst_records.apply(
    lambda row: row.reconstructed_probability_vector[row.largest_task_index],
    axis=1,
)
worst_records[[
    "model_id", "ecg_key", "patient_key", "sex", "age_at_ecg",
    "location_setting", "probability_drift", "largest_task",
    "largest_task_drift", "reference_probability_on_largest_task",
    "reconstructed_probability_on_largest_task", "pearson", "rmse",
]]

,model_id,ecg_key,patient_key,sex,age_at_ecg,location_setting,probability_drift,largest_task,largest_task_drift,reference_probability_on_largest_task,reconstructed_probability_on_largest_task,pearson,rmse
24147,ecgaim__e1c0m0d0__s42,6677264,568415965,female,78,inpatient,0.536053,pulmonary_regurgitation_moderate_or_greater,0.772458,0.058118,0.830576,0.170969,0.037410
23257,ecgaim__e1c0m0d0__s42,7215435,7382292343,female,66,inpatient,0.508839,pulmonary_regurgitation_moderate_or_greater,0.739892,0.856132,0.116240,0.565595,0.027624
23315,ecgaim__e1c0m0d0__s42,11173398,226963591,male,79,emergency,0.469347,lvef_lte_45,0.615153,0.750894,0.135741,0.470512,0.037719
28699,ecgaim__e1c1m1d1__s42,7215435,7382292343,female,66,inpatient,0.537689,pulmonary_regurgitation_moderate_or_greater,0.769382,0.856132,0.086751,0.580239,0.027574
29589,ecgaim__e1c1m1d1__s42,6677264,568415965,female,78,inpatient,0.509302,pulmonary_regurgitation_moderate_or_greater,0.741338,0.058118,0.799456,0.261981,0.037128
32309,ecgaim__e1c1m1d1__s42,10127945,7591575947,male,69,inpatient,0.473163,pulmonary_regurgitation_moderate_or_greater,0.616314,0.652845,0.036532,0.766693,0.026389
13263,msvae__e1c0m0d0__s42,6677264,568415965,female,78,inpatient,0.621085,pulmonary_regurgitation_moderate_or_greater,0.858597,0.058118,0.916715,0.235051,0.061592
12507,msvae__e1c0m0d0__s42,7022266,9195047281,female,37,emergency,0.538919,shd_moderate_or_greater,0.696762,0.077031,0.773793,0.711199,0.039163
16069,msvae__e1c0m0d0__s42,7505797,2325867453,female,61,procedural,0.523814,shd_moderate_or_greater,0.667003,0.202146,0.869148,0.715514,0.049996
18705,msvae__e1c1m1d1__s42,6677264,568415965,female,78,inpatient,0.633788,pulmonary_regurgitation_moderate_or_greater,0.865589,0.058118,0.923707,0.135725,0.064689


In [13]:
#| label: echonext-morphology-diagnostic-discordance
#| tbl-cap: Association and discordant tails between missing-lead morphology and classifier drift.
from scipy.stats import spearmanr

discordance_rows = []
for model_id, group in record_drift.groupby("model_id", sort=False):
    rho, p_value = spearmanr(
        group.pearson, group.probability_drift, nan_policy="omit"
    )
    high_drift = group.probability_drift >= group.probability_drift.quantile(0.90)
    morphology_median = group.pearson.median()
    discordance_rows.append({
        "model_id": model_id,
        "records": len(group),
        "Spearman_rho_Pearson_vs_drift": rho,
        "nominal_p_value": p_value,
        "morphology_Pearson_median": morphology_median,
        "drift_p90": group.probability_drift.quantile(0.90),
        "high_morphology_high_drift_records": (
            high_drift & (group.pearson >= morphology_median)
        ).sum(),
        "low_morphology_high_drift_records": (
            high_drift & (group.pearson < morphology_median)
        ).sum(),
    })
pd.DataFrame(discordance_rows)

,model_id,records,Spearman_rho_Pearson_vs_drift,nominal_p_value,morphology_Pearson_median,drift_p90,high_morphology_high_drift_records,low_morphology_high_drift_records
0,unet__e1c0m0d0__s42,5442,0.153635,4.239575e-30,0.804185,0.296217,302,243
1,unet__e1c1m1d1__s42,5442,0.185343,2.957751e-43,0.907567,0.326965,298,247
2,msvae__e1c0m0d0__s42,5442,0.074823,3.272365e-08,0.831583,0.230873,266,279
3,msvae__e1c1m1d1__s42,5442,0.181455,1.672586e-41,0.902147,0.316359,283,262
4,ecgaim__e1c0m0d0__s42,5442,0.038826,4.175674e-03,0.896511,0.189475,283,262
5,ecgaim__e1c1m1d1__s42,5442,0.094222,3.292822e-12,0.940338,0.188113,295,250
